# Célula 1 — Markdown
"""
## 3. Methodology

### 3.1 Evaluation Protocol
Para cada amostra do subconjunto, executamos inferência com ambos os modelos
e coletamos: transcrição/tradução gerada, WER, BLEU e latência de inferência.
"""

In [3]:
import torch
from transformers import SeamlessM4Tv2ForSpeechToText, AutoProcessor

model_name = "facebook/seamless-m4t-v2-large"

processor = AutoProcessor.from_pretrained(model_name)
model = SeamlessM4Tv2ForSpeechToText.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Modelo carregado em:", device)

c:\Users\david\miniconda3\envs\asr-benchmark\lib\site-packages\transformers\deepspeed.py:23: FutureWarning: transformers.deepspeed module is deprecated and will be removed in a future version. Please import deepspeed modules directly from transformers.integrations
  warnings.warn(
c:\Users\david\miniconda3\envs\asr-benchmark\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Modelo carregado em: cuda


In [4]:
from datasets import load_dataset

data_dir = "../data/1781716768543-cv-corpus-26.0-2026-06-12-pt/cv-corpus-26.0-2026-06-12/pt"

dataset = load_dataset("facebook/covost2", "pt_en", data_dir=data_dir, split="test")
print(f"Total de amostras: {len(dataset)}")

c:\Users\david\miniconda3\envs\asr-benchmark\lib\site-packages\datasets\load.py:1461: FutureWarning: The repository for facebook/covost2 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/facebook/covost2
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Total de amostras: 4023


In [5]:
import torch

exemplo = dataset[0]
audio_array = exemplo['audio']['array']
sampling_rate = exemplo['audio']['sampling_rate']

print("Frase original (pt):", exemplo['sentence'])
print("Tradução esperada (en):", exemplo['translation'])

# Processar o áudio
inputs = processor(audios=audio_array, sampling_rate=sampling_rate, return_tensors="pt").to(device)

# Gerar tradução (pt -> en)
with torch.no_grad():
    output_tokens = model.generate(**inputs, tgt_lang="eng")

texto_traduzido = processor.decode(output_tokens[0].tolist(), skip_special_tokens=True)
print("SeamlessM4T:", texto_traduzido)

Frase original (pt): Pedir dinheiro emprestado às pessoas da aldeia
Tradução esperada (en): Borrow money from people in the village
SeamlessM4T: Asking people in the village to lend money


In [6]:
import time
import pandas as pd
import os

resultados = []
checkpoint_path = '../results/seamless_resultados_parcial.csv'
os.makedirs('../results', exist_ok=True)

total = len(dataset)

for i in range(total):
    exemplo = dataset[i]
    audio_array = exemplo['audio']['array']
    sampling_rate = exemplo['audio']['sampling_rate']
    referencia = exemplo['translation']
    sentenca_pt = exemplo['sentence']

    inicio = time.time()

    inputs = processor(audios=audio_array, sampling_rate=sampling_rate, return_tensors="pt").to(device)
    with torch.no_grad():
        output_tokens = model.generate(**inputs, tgt_lang="eng")
    texto_traduzido = processor.decode(output_tokens[0].tolist(), skip_special_tokens=True)

    fim = time.time()

    resultados.append({
        'id': exemplo['id'],
        'sentence_pt': sentenca_pt,
        'reference_en': referencia,
        'seamless_translation': texto_traduzido,
        'latency_sec': fim - inicio
    })

    if i % 100 == 0:
        print(f"{i}/{total} processados...")
        pd.DataFrame(resultados).to_csv(checkpoint_path, index=False)

df_resultados = pd.DataFrame(resultados)
df_resultados.to_csv('../results/seamless_resultados_final.csv', index=False)
print("Concluído! Resultados salvos em results/seamless_resultados_final.csv")
df_resultados.head()

0/4023 processados...
100/4023 processados...
200/4023 processados...
300/4023 processados...
400/4023 processados...
500/4023 processados...
600/4023 processados...
700/4023 processados...
800/4023 processados...
900/4023 processados...
1000/4023 processados...
1100/4023 processados...
1200/4023 processados...
1300/4023 processados...
1400/4023 processados...
1500/4023 processados...
1600/4023 processados...
1700/4023 processados...
1800/4023 processados...
1900/4023 processados...
2000/4023 processados...
2100/4023 processados...
2200/4023 processados...
2300/4023 processados...
2400/4023 processados...
2500/4023 processados...
2600/4023 processados...
2700/4023 processados...
2800/4023 processados...
2900/4023 processados...
3000/4023 processados...
3100/4023 processados...
3200/4023 processados...
3300/4023 processados...
3400/4023 processados...
3500/4023 processados...
3600/4023 processados...
3700/4023 processados...
3800/4023 processados...
3900/4023 processados...
4000/4023 pr

,id,sentence_pt,reference_en,seamless_translation,latency_sec
0,common_voice_pt_19484624,Pedir dinheiro emprestado às pessoas da aldeia,Borrow money from people in the village,Asking people in the village to lend money,1.478034
1,common_voice_pt_19405032,Trancá-los,Lock them up,Turn off the bus.,0.930890
2,common_voice_pt_19522569,O Youtube ainda é a melhor plataforma de vídeos.,Youtube is still the best video platform.,YouTube is still the best video platform.,1.035351
3,common_voice_pt_19665686,Menina e menino beijando nas sombras,A girl and a boy kissing in the shadows,Girl and boy kissing in the shadows.,2.066535
4,common_voice_pt_19854926,O mago lançou um feitiço muito poderoso sobre ...,The wizard cast a very powerful spell on the c...,The magician has unleashed a very powerful spe...,2.289127


In [ ]:
# Célula 2 — Funções de métricas
from jiwer import wer as calcular_wer
from sacrebleu.metrics import BLEU
import time

bleu_metric = BLEU(effective_order=True)

def avaliar_amostra(hipotese_transcricao, hipotese_traducao,
                    referencia_transcricao, referencia_traducao,
                    tempo_inferencia, duracao_audio):
    return {
        "WER":      calcular_wer(referencia_transcricao, hipotese_transcricao),
        "BLEU":     bleu_metric.sentence_score(hipotese_traducao, [referencia_traducao]).score,
        "latencia_s":   tempo_inferencia,
        "RTF":      tempo_inferencia / duracao_audio,  # Real-Time Factor — métrica chave
    }

In [ ]:
# Célula 3 — Inferência SeamlessM4T
import torch
import torchaudio

resultados_seamless = []

for amostra in subset.select(range(50)):  # começa com 50 para testar
    audio = torch.tensor(amostra['audio']['array']).unsqueeze(0).float()
    sr    = amostra['audio']['sampling_rate']
    
    inputs = processor_seamless(
        audios=audio, sampling_rate=sr,
        src_lang="por", tgt_lang="eng",
        return_tensors="pt"
    )
    
    t0 = time.perf_counter()
    with torch.no_grad():
        output_tokens = model_seamless.generate(**inputs, tgt_lang="eng")
    latencia = time.perf_counter() - t0
    
    traducao = processor_seamless.decode(output_tokens[0].tolist(), skip_special_tokens=True)
    duracao  = len(amostra['audio']['array']) / sr
    
    metricas = avaliar_amostra(
        hipotese_transcricao=traducao,   # SeamlessM4T faz S2TT direto
        hipotese_traducao=traducao,
        referencia_transcricao=amostra['sentence'],
        referencia_traducao=amostra['translation'],
        tempo_inferencia=latencia,
        duracao_audio=duracao
    )
    metricas['modelo'] = 'SeamlessM4T-v2'
    resultados_seamless.append(metricas)

print(f"SeamlessM4T — {len(resultados_seamless)} amostras processadas")

In [ ]:
# Célula 4 — Inferência NeMo Canary
resultados_canary = []

model_canary.eval()

for amostra in subset.select(range(50)):
    audio_path = f"/tmp/amostra_{amostra['id']}.wav"
    torchaudio.save(audio_path,
                    torch.tensor(amostra['audio']['array']).unsqueeze(0),
                    amostra['audio']['sampling_rate'])
    duracao = len(amostra['audio']['array']) / amostra['audio']['sampling_rate']

    t0 = time.perf_counter()
    resultado = model_canary.transcribe(
        audio=[audio_path],
        batch_size=1,
        task="s2t_translation",
        source_lang="pt",
        target_lang="en"
    )
    latencia = time.perf_counter() - t0

    metricas = avaliar_amostra(
        hipotese_transcricao=resultado[0].text,
        hipotese_traducao=resultado[0].text,
        referencia_transcricao=amostra['sentence'],
        referencia_traducao=amostra['translation'],
        tempo_inferencia=latencia,
        duracao_audio=duracao
    )
    metricas['modelo'] = 'NeMo Canary-1B'
    resultados_canary.append(metricas)

In [ ]:
# Célula 5 — Consolidação e Table II do artigo
df = pd.DataFrame(resultados_seamless + resultados_canary)

tabela_resultados = df.groupby('modelo').agg(
    WER_medio=('WER', 'mean'),
    BLEU_medio=('BLEU', 'mean'),
    Latencia_media_s=('latencia_s', 'mean'),
    RTF_medio=('RTF', 'mean')
).round(4)

print(tabela_resultados)
tabela_resultados.to_csv("results/table2_main_results.csv")
# → Esta saída é a Table II do artigo

In [ ]:
# Célula 6 — Figura 2: comparação visual (Figure 2 do artigo)
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
metricas_plot = ['WER', 'BLEU', 'RTF']
titulos       = ['WER ↓ (menor = melhor)', 'BLEU ↑ (maior = melhor)', 'RTF ↓ (tempo real)']
cores         = {'SeamlessM4T-v2': 'steelblue', 'NeMo Canary-1B': 'coral'}

for ax, metrica, titulo in zip(axes, metricas_plot, titulos):
    for modelo, grupo in df.groupby('modelo'):
        ax.hist(grupo[metrica], alpha=0.65, label=modelo,
                color=cores[modelo], bins=20, edgecolor='white')
    ax.set_title(titulo, fontsize=11)
    ax.set_xlabel(metrica)
    ax.legend(fontsize=8)

plt.suptitle("Distribuição de métricas por modelo — CoVoST-2 pt→en", fontsize=12)
plt.tight_layout()
plt.savefig("figures/fig2_metrics_comparison.png", dpi=150, bbox_inches='tight')
plt.show()